# 과제 2 · 지난 5년간 서울시 전세가격 변화
2021–2025년 계약, ㎡당 전세보증금 중앙값을 주 지표로 사용합니다. 원본 출처: 서울 열린데이터광장 OA-21276.
전처리 전체 코드는 `scripts/analyze.py`, 결과 해석과 한계는 `ANALYSIS.md`, `README.md`에 있습니다.
최초 재현: `pip install -r requirements.txt` → `python scripts/download_data.py` → `python scripts/analyze.py`. 아래는 생성된 집계를 이용한 Plotly Express 분석입니다.

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import json
annual = pd.read_csv('public/data/annual.csv')
monthly = pd.read_csv('public/data/monthly.csv')
city = annual.query('district == "서울 전체" and type == "전체 유형"').set_index('year')
city[['n', 'median_m2', 'mean_m2', 'median_deposit']]

## 1. 월별 흐름
단위는 만원/㎡입니다. 월별 중앙값을 사용해 극단값 영향을 줄이고, 거래 건수를 함께 확인합니다.

In [ ]:
m = monthly.query('district == "서울 전체" and type == "전체 유형"')
fig = px.line(m, x='month', y='median_m2', markers=True, hover_data=['n'], labels={'month':'계약월', 'median_m2':'중앙값 (만원/㎡)'}, title='서울시 전세가격의 월별 변화')
fig.update_layout(template='plotly_white')
fig.show()
# fig.write_html('monthly.html')  # 단독 HTML로 저장하려면 실행

## 2. 자치구별 변화율
2021년 대비 2025년의 연간 대표값 변화입니다. 같은 주택의 가격 변동률이 아니라 그 지역에서 신고된 계약의 대표값 비교입니다.

In [ ]:
gu = annual.query('district != "서울 전체" and type == "전체 유형"').pivot(index='district', columns='year', values='median_m2')
change = ((gu[2025] / gu[2021] - 1) * 100).sort_values().rename('change').reset_index()
fig = px.bar(change, x='change', y='district', orientation='h', labels={'change':'변화율 (%)','district':'자치구'}, title='자치구별 ㎡당 중앙값 변화 · 2021→2025')
fig.update_layout(template='plotly_white', height=750)
fig.show()

## 3. 주택유형별 추세
아파트·연립다세대·단독다가구·오피스텔을 나누어 전체 지표와 유형별 흐름을 비교합니다.

In [ ]:
types = annual.query('district == "서울 전체" and type != "전체 유형"')
fig = px.line(types, x='year', y='median_m2', color='type', markers=True, hover_data=['n'], labels={'year':'계약연도','median_m2':'중앙값 (만원/㎡)','type':'주택유형'})
fig.update_layout(template='plotly_white')
fig.update_xaxes(dtick=1)
fig.show()

## 결과 해석
아래 문장은 집계값으로 자동 생성됩니다. 전체 상승률뿐 아니라 유형별 차이와 거래 구성 변화를 함께 해석합니다.

In [ ]:
result = json.loads(Path('public/data/analysis.json').read_text(encoding='utf-8'))
for finding in result['findings']:
    print(finding['title'])
    print(finding['text'], '\n')

## 해석의 한계
2026년 접수분에 포함된 2025년 계약은 제외됩니다. 신고 지연·수록 범위·정정 시점·신규/갱신 및 주택 구성 변화가 영향을 줄 수 있습니다. 2024년과 2025년 공개 파일 행 수가 크게 달라 계약 건수 증가를 시장 전체 거래량 증가로 단정하지 않습니다. 금리나 정책의 인과효과를 추정하지 않았습니다. 원본 파일은 공공누리 제1유형(출처표시)이며, 원본 체크섬은 `data/sources.json`에 기록되어 있습니다.